# Standalone Setup & Ingest (own AWS account)

Sets up everything this project needs inside **whichever AWS account this notebook runs in**,
with no dependency on another account's bucket, Glue database, Athena tables, or config file.

What it does, in order:
1. Detects the current account, region and SageMaker execution role.
2. Finds a usable S3 bucket. Some AWS accounts (for example learner/sandbox accounts) block
   `s3:CreateBucket` for custom names, so it tries the SageMaker default bucket first
   (`sagemaker-<region>-<account-id>`), then a project-named bucket, and uses the first one that works.
3. Creates a Glue database and an Athena workgroup (best effort; if the account forbids them, the
   notebooks that follow fall back to reading S3 directly).
4. Downloads PAMAP2 from UCI on the notebook instance itself (the job network is often more
   locked down than the instance), lands the raw `.dat` files in S3, and writes curated
   Parquet partitioned by session and subject. Only the files actually needed are kept on
   local disk, and each one is deleted right after it is uploaded.
5. Saves `config/project_config.json` in the format the other notebooks read, plus a small
   subject-by-activity coverage table used for exploratory analysis.

Run this notebook once, then run `A_data_feature_engineering.ipynb` and
`B_model_training_deployment_monitoring.ipynb`.

## 1. Environment check

In [1]:
from importlib.metadata import version, PackageNotFoundError

try:
    _sm_version = version("sagemaker")
except PackageNotFoundError:
    _sm_version = "0"

# The notebooks use the SageMaker Python SDK v2 API (Session, FrameworkProcessor, Estimator.deploy).
if int(_sm_version.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3" requests pyarrow
    print(f"Replaced sagemaker {_sm_version} with v2. Restarting the kernel: re-run this notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    %pip install -q requests pyarrow
    print("sagemaker", _sm_version)

Note: you may need to restart the kernel to use updated packages.
sagemaker 2.257.6


## 2. Account, region and execution role

In [2]:
import sys, pathlib, json, io, time, zipfile, shutil

PROJECT_ROOT = pathlib.Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import boto3
import requests
import pandas as pd
import sagemaker
from botocore.exceptions import ClientError

from src.pamap2.config import build_config, save_config
from src.pamap2 import schema

boto_session = boto3.Session()
REGION = boto_session.region_name or "us-east-1"
identity = boto_session.client("sts").get_caller_identity()
ACCOUNT_ID = identity["Account"]
EXECUTION_ROLE = sagemaker.get_execution_role()

print("Account:", ACCOUNT_ID)
print("Region: ", REGION)
print("Role:   ", EXECUTION_ROLE)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Account: 836620944282
Region:  ap-southeast-2
Role:    arn:aws:iam::836620944282:role/service-role/AmazonSageMaker-ExecutionRole-20261015T160795


## 3. Find a bucket this account is allowed to use

Set `EXISTING_BUCKET` to a bucket name you already have access to if you have one (a bucket an
administrator created for you, for example). Leave it empty to let the notebook try the SageMaker
default bucket and then a project-named bucket.

In [3]:
EXISTING_BUCKET = ""   # optional: a bucket you already have read/write access to

s3 = boto_session.client("s3", region_name=REGION)


def bucket_usable(name):
    """True if the bucket exists (or can be created) and this identity can write to it."""
    try:
        s3.head_bucket(Bucket=name)
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        if code_ not in ("404", "NoSuchBucket", "NotFound"):
            print(f"  {name}: not accessible ({code_})")
            return False
        try:
            if REGION == "us-east-1":
                s3.create_bucket(Bucket=name)
            else:
                s3.create_bucket(Bucket=name, CreateBucketConfiguration={"LocationConstraint": REGION})
            print(f"  {name}: created")
        except ClientError as e2:
            print(f"  {name}: cannot create ({e2.response['Error']['Code']})")
            return False
    try:
        s3.put_object(Bucket=name, Key="pamap2-har/.write-test", Body=b"ok")
        s3.delete_object(Bucket=name, Key="pamap2-har/.write-test")
    except ClientError as e:
        print(f"  {name}: no write access ({e.response['Error']['Code']})")
        return False
    return True


candidates = []
if EXISTING_BUCKET:
    candidates.append(EXISTING_BUCKET)
try:
    candidates.append(sagemaker.Session(boto_session=boto_session).default_bucket())
except Exception as e:
    print("SageMaker default bucket unavailable:", type(e).__name__)
candidates.append(f"pamap2-har-{ACCOUNT_ID}-{REGION}")

print("Trying buckets in order:", candidates)
BUCKET = next((b for b in candidates if bucket_usable(b)), None)
if BUCKET is None:
    raise RuntimeError(
        "No usable S3 bucket. This account blocks bucket creation and none of the candidates is "
        "accessible. Ask an administrator for a bucket, then set EXISTING_BUCKET above and re-run.")
print("Using bucket:", BUCKET)

Trying buckets in order: ['sagemaker-ap-southeast-2-836620944282', 'pamap2-har-836620944282-ap-southeast-2']
Using bucket: sagemaker-ap-southeast-2-836620944282


## 4. Project config for this account

In [4]:
cfg = build_config(ACCOUNT_ID, region=REGION)
cfg["bucket"] = BUCKET
cfg["role_arn"] = EXECUTION_ROLE
print(json.dumps({k: cfg[k] for k in ("region", "bucket", "role_arn", "glue_database", "athena_workgroup")}, indent=2))

for prefix in cfg["s3_prefixes"].values():
    try:
        s3.put_object(Bucket=BUCKET, Key=prefix)
    except ClientError as e:
        print("prefix", prefix, "->", e.response["Error"]["Code"])

# Optional hardening; skipped quietly if the account does not allow it.
for label, call in (
    ("public access block", lambda: s3.put_public_access_block(Bucket=BUCKET, PublicAccessBlockConfiguration={
        "BlockPublicAcls": True, "IgnorePublicAcls": True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True})),
    ("default encryption", lambda: s3.put_bucket_encryption(Bucket=BUCKET, ServerSideEncryptionConfiguration={
        "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]})),
):
    try:
        call()
        print("applied", label)
    except ClientError as e:
        print("skipped", label, "->", e.response["Error"]["Code"])

config_path = save_config(cfg)
print("Wrote", config_path)

{
  "region": "ap-southeast-2",
  "bucket": "sagemaker-ap-southeast-2-836620944282",
  "role_arn": "arn:aws:iam::836620944282:role/service-role/AmazonSageMaker-ExecutionRole-20261015T160795",
  "glue_database": "pamap2_har",
  "athena_workgroup": "pamap2-har-wg"
}
skipped public access block -> AccessDenied
skipped default encryption -> AccessDenied
Wrote /home/ec2-user/SageMaker/Wearable-Sensor-Based-Sports-Activity-Recognition/config/project_config.json


## 5. Glue database and Athena workgroup (best effort)

Athena makes the exploratory queries and Feature Store reads convenient. If the account does not
permit these services, the notebook keeps going and the later notebooks read S3 directly instead.

In [5]:
ATHENA_OK = False
try:
    glue = boto_session.client("glue", region_name=REGION)
    try:
        glue.create_database(DatabaseInput={"Name": cfg["glue_database"], "Description": "PAMAP2 HAR data lake"})
        print("created Glue database", cfg["glue_database"])
    except glue.exceptions.AlreadyExistsException:
        print("Glue database exists:", cfg["glue_database"])

    athena = boto_session.client("athena", region_name=REGION)
    try:
        athena.create_work_group(
            Name=cfg["athena_workgroup"], Description="PAMAP2 HAR queries",
            Configuration={"ResultConfiguration": {
                "OutputLocation": f"s3://{BUCKET}/{cfg['s3_prefixes']['athena-results']}",
                "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"}},
                "EnforceWorkGroupConfiguration": True})
        print("created Athena workgroup", cfg["athena_workgroup"])
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidRequestException" and "already created" in e.response["Error"]["Message"]:
            print("Athena workgroup exists:", cfg["athena_workgroup"])
        else:
            raise
    ATHENA_OK = True
except Exception as e:
    print("Glue/Athena setup skipped:", type(e).__name__, str(e)[:200])
    print("Continuing without Athena; later notebooks fall back to S3 reads.")

Glue database exists: pamap2_har
Athena workgroup exists: pamap2-har-wg


## 6. Download PAMAP2 from UCI

Runs on this notebook's own instance, not as a SageMaker job: the job network (used by Processing
and Training jobs) is often more locked down than the notebook instance itself, and the instance is
usually the one with a route to the public internet. About 650 MB total; only the nine
`subject*.dat` files are kept, everything else is deleted as soon as it is used, so this comfortably
fits even on a small instance's disk.

If this cell times out or fails to connect, the notebook instance itself has no outbound internet
access either, and the download needs to happen from somewhere that does (a local machine) with the
resulting raw files uploaded to `s3://<bucket>/raw/pamap2/` by hand.

In [6]:
def pick_scratch():
    """Use whichever local volume has the most free space (the root volume is sometimes small)."""
    volumes = [p for p in (pathlib.Path("/home/ec2-user/SageMaker"), pathlib.Path("/tmp")) if p.exists()]
    return max(volumes, key=lambda p: shutil.disk_usage(p).free) / "pamap2_scratch"


SCRATCH = pick_scratch()
for leftover in (pathlib.Path("/tmp/pamap2"), SCRATCH):   # clear any earlier failed run
    shutil.rmtree(leftover, ignore_errors=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
print(f"Scratch: {SCRATCH}  ({shutil.disk_usage(SCRATCH).free / 1e9:.1f} GB free)")

URL = "https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip"
zip_path = SCRATCH / "pamap2.zip"

with requests.get(URL, stream=True, timeout=120) as response:
    response.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1 << 20):
            f.write(chunk)
print(f"Downloaded {zip_path.stat().st_size / 1e6:.0f} MB")

DAT_DIR = SCRATCH / "dat"


def extract_subject_files(archive):
    """Extract only subject*.dat (keeping the Protocol/Optional folder), looking inside nested zips."""
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            name = member.lower()
            if name.endswith(".dat") and "subject" in name:
                z.extract(member, DAT_DIR)
            elif name.endswith(".zip"):               # the UCI archive nests PAMAP2_Dataset.zip
                nested = z.extract(member, SCRATCH / "nested")
                extract_subject_files(nested)
                pathlib.Path(nested).unlink()


extract_subject_files(zip_path)
zip_path.unlink()                                       # free the space before ingesting
shutil.rmtree(SCRATCH / "nested", ignore_errors=True)

dat_files = sorted(DAT_DIR.rglob("subject*.dat"))
print(len(dat_files), "subject files:", [f.parent.name + "/" + f.name for f in dat_files])
print(f"{shutil.disk_usage(SCRATCH).free / 1e9:.1f} GB free after extraction")

Scratch: /home/ec2-user/SageMaker/pamap2_scratch  (4.9 GB free)
Downloaded 688 MB
14 subject files: ['Optional/subject101.dat', 'Optional/subject105.dat', 'Optional/subject106.dat', 'Optional/subject108.dat', 'Optional/subject109.dat', 'Protocol/subject101.dat', 'Protocol/subject102.dat', 'Protocol/subject103.dat', 'Protocol/subject104.dat', 'Protocol/subject105.dat', 'Protocol/subject106.dat', 'Protocol/subject107.dat', 'Protocol/subject108.dat', 'Protocol/subject109.dat']
3.2 GB free after extraction


## 7. Land raw files and write curated Parquet

Also runs on the notebook instance (it is only parsing small per-file batches, not holding the
whole dataset in memory). Raw `.dat` files are uploaded byte-for-byte unchanged. Curated Parquet
drops the four invalid orientation columns per IMU and is partitioned by `session` and
`subject_id`. A subject-by-activity row-count table is written alongside it for later exploratory
analysis. Each local file is deleted right after it is uploaded.

In [7]:
def s3_object_size(key):
    try:
        return s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
            raise
        return None


coverage_rows = []
total_rows = 0
for dat in dat_files:
    session_name = dat.parent.name.lower()               # protocol | optional
    subject_id = int(dat.stem.replace("subject", ""))

    raw_key = f"{cfg['s3_prefixes']['raw']}pamap2/{session_name}/{dat.name}"
    if s3_object_size(raw_key) != dat.stat().st_size:
        s3.upload_file(str(dat), BUCKET, raw_key)

    frame = schema.parse_dat(dat, subject_id=subject_id, session=session_name)
    curated_local = SCRATCH / f"{session_name}_{subject_id}.parquet"
    frame.drop(columns=["subject_id", "session"]).to_parquet(curated_local, index=False)
    curated_key = (f"{cfg['s3_prefixes']['curated']}pamap2/session={session_name}/"
                   f"subject_id={subject_id}/part-0.parquet")
    s3.upload_file(str(curated_local), BUCKET, curated_key)
    curated_local.unlink()

    per_activity = frame[frame["activity_id"] != 0].groupby("activity_id").size()
    for activity_id, n_rows in per_activity.items():
        coverage_rows.append({"subject_id": subject_id, "activity_id": int(activity_id), "n_rows": int(n_rows)})

    total_rows += len(frame)
    print(f"{session_name}/{subject_id}: {len(frame):,} rows")
    dat.unlink()

print(f"Total rows: {total_rows:,}")

coverage = (pd.DataFrame(coverage_rows).groupby(["subject_id", "activity_id"], as_index=False)["n_rows"].sum())
coverage_key = f"{cfg['s3_prefixes']['curated']}coverage.csv"
s3.put_object(Bucket=BUCKET, Key=coverage_key, Body=coverage.to_csv(index=False).encode())
print("Wrote coverage table to", f"s3://{BUCKET}/{coverage_key}")

shutil.rmtree(SCRATCH, ignore_errors=True)               # everything is in S3 now
print("Local scratch cleared")

optional/101: 319,352 rows
optional/105: 154,773 rows
optional/106: 129,963 rows
optional/108: 180,412 rows
optional/109: 193,472 rows
protocol/101: 376,417 rows
protocol/102: 447,000 rows
protocol/103: 252,833 rows
protocol/104: 329,576 rows
protocol/105: 374,783 rows
protocol/106: 361,817 rows
protocol/107: 313,599 rows
protocol/108: 408,031 rows
protocol/109: 8,477 rows
Total rows: 3,850,505
Wrote coverage table to s3://sagemaker-ap-southeast-2-836620944282/curated/coverage.csv
Local scratch cleared


## 8. Register Athena tables (only if Athena is available)

In [8]:
def athena_run(sql, timeout_s=600):
    query_id = athena.start_query_execution(
        QueryString=sql, WorkGroup=cfg["athena_workgroup"],
        QueryExecutionContext={"Database": cfg["glue_database"]})["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while True:
        state = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]["Status"]
        if state["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            raise TimeoutError(sql[:100])
        time.sleep(1)
    if state["State"] != "SUCCEEDED":
        raise RuntimeError(f"{state['State']}: {state.get('StateChangeReason')}")


def column_ddl(columns):
    return ",\n  ".join(f"`{c}` {'int' if c == 'activity_id' else 'double'}" for c in columns)


if ATHENA_OK:
    try:
        raw_location = f"s3://{BUCKET}/{cfg['s3_prefixes']['raw']}pamap2/"
        curated_location = f"s3://{BUCKET}/{cfg['s3_prefixes']['curated']}pamap2/"
        athena_run(f"""CREATE EXTERNAL TABLE IF NOT EXISTS {cfg['raw_table']} (
  {column_ddl(schema.COLUMNS)}
) ROW FORMAT DELIMITED FIELDS TERMINATED BY ' ' STORED AS TEXTFILE LOCATION '{raw_location}'""")
        athena_run(f"""CREATE EXTERNAL TABLE IF NOT EXISTS {cfg['curated_table']} (
  {column_ddl(schema.CURATED_COLUMNS)}
) PARTITIONED BY (session string, subject_id int) STORED AS PARQUET LOCATION '{curated_location}'""")
        athena_run(f"MSCK REPAIR TABLE {cfg['curated_table']}")
        print("Athena tables ready:", cfg["raw_table"], cfg["curated_table"])
    except Exception as e:
        print("Athena table registration skipped:", type(e).__name__, str(e)[:200])
else:
    print("Athena not available; skipping table registration.")

Athena tables ready: raw_pamap2 curated_pamap2


## Summary

- Bucket in use, role, region and resource names are saved in `config/project_config.json`.
- Raw `.dat` files are under `raw/pamap2/`, curated Parquet under `curated/pamap2/`, and the
  subject-by-activity coverage table at `curated/coverage.csv`.
- Next: `A_data_feature_engineering.ipynb`, then `B_model_training_deployment_monitoring.ipynb`.